<a href="https://colab.research.google.com/github/NajmehNayyer/NOD-EEG/blob/main/epoching_and_concatenating_all_subjects.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Choosing Subjects

In [ ]:
first = range(1,7)
second  = range(6, 11)
third  = range(11, 21)
forth = range(21, 31)

In [ ]:
print("which set of subjects do you want?")
input_ids_str = input("Type your response: ")

if input_ids_str == "first":
    input_ids = first
    estimeted_n_epochs = 20000

elif input_ids_str == "second":
    input_ids = second
    estimeted_n_epochs = 18000

elif input_ids_str == "third":
    input_ids = third
    estimeted_n_epochs = 20000

elif input_ids_str == "forth":
    input_ids = forth
    estimeted_n_epochs = 20000

subject_ids = [f'{i:02d}' for i in input_ids if i != 3]
print(f'IDs: {subject_ids}')

which set of subjects do you want?
Type your response: first
IDs: ['01', '02', '04', '05', '06']


# Importing Data and Libraries

In [ ]:
import os
import warnings
import re

from google.colab import files

import numpy as np
import pandas as pd

try:
    import mne
except ImportError:
    !pip install mne
    import mne

In [ ]:
!pip install awscli

In [ ]:
include_raw = ' '.join([f'--include "derivatives/preprocessed/raw/sub-{sid}_*"' for sid in subject_ids])
include_events = ' '.join([f'--include "derivatives/detailed_events/sub-{sid}_events.csv"' for sid in subject_ids])

!aws s3 sync --no-sign-request s3://openneuro.org/ds005811 ./ds005811/ --exclude "*" \
{include_raw} \
{include_events}

%cd ds005811

download: s3://openneuro.org/ds005811/derivatives/detailed_events/sub-05_events.csv to ds005811/derivatives/detailed_events/sub-05_events.csv
download: s3://openneuro.org/ds005811/derivatives/detailed_events/sub-02_events.csv to ds005811/derivatives/detailed_events/sub-02_events.csv
download: s3://openneuro.org/ds005811/derivatives/detailed_events/sub-01_events.csv to ds005811/derivatives/detailed_events/sub-01_events.csv
download: s3://openneuro.org/ds005811/derivatives/detailed_events/sub-06_events.csv to ds005811/derivatives/detailed_events/sub-06_events.csv
download: s3://openneuro.org/ds005811/derivatives/detailed_events/sub-04_events.csv to ds005811/derivatives/detailed_events/sub-04_events.csv
download: s3://openneuro.org/ds005811/derivatives/preprocessed/raw/sub-01_ses-ImageNet01_task-ImageNet_run-02_eeg_clean.fif to ds005811/derivatives/preprocessed/raw/sub-01_ses-ImageNet01_task-ImageNet_run-02_eeg_clean.fif
download: s3://openneuro.org/ds005811/derivatives/preprocessed/raw/s

# Info

In [ ]:
# raw = mne.io.read_raw_fif("/content/ds005811/derivatives/preprocessed/raw/sub-01_ses-ImageNet01_task-ImageNet_run-01_eeg_clean.fif")
# info = raw.info

# Data

In [ ]:
def sort_key(filename):
    session_match = re.search(r'ses-ImageNet(\d+)', filename)
    run_match = re.search(r'run-(\d+)', filename)
    session_num = int(session_match.group(1)) if session_match else 0
    run_num = int(run_match.group(1)) if run_match else 0
    return (session_num, run_num)

In [ ]:
def get_data_of_subject(subject_id):

    # Turn off the warnings
    warnings.filterwarnings("ignore", module="mne")

    # Prep
    run_counter = 0
    data_path = "/content/ds005811/derivatives/preprocessed/raw/"
    metadata_path = f"/content/ds005811/derivatives/detailed_events/sub-{subject_id}_events.csv"

    # Print Subject ID
    print("="*60)
    print(f'Subjcet {subject_id}')
    print("="*60)

    # Collect the filenames
    fif_files = []
    for filename in os.listdir(data_path):
        if filename.startswith(f"sub-{subject_id}") and filename.endswith(".fif"):
            # Add to list
            fif_files.append(filename)
    # Sort
    fif_files.sort(key=sort_key)

    # Collect files
    all_epochs = []
    for filename in fif_files:

        run_counter += 1
        file_path = os.path.join(data_path, filename)

        try:
            # Data of each run
            raw = mne.io.read_raw_fif(file_path, preload=True, verbose=False)

            # extract events
            events, event_id_map = mne.events_from_annotations(raw, verbose=False)

            # Define event_id
            stim_on_id = event_id_map.get('stim_on')
            if stim_on_id is None:
                print(f"Warning: 'stim_on' event not found in annotations for {filename}. Skipping epoching for this file.")
                continue

            event_id = {'stim_on': stim_on_id}

            # Epoch without baseline
            epochs = mne.Epochs(raw, events, event_id=event_id, tmin=-0.1, tmax=0.8,
                                preload=True, baseline=None, verbose=False)

            # Add to all data
            all_epochs.append(epochs)

        except Exception as e:
            print(f"Could not process file {filename} for subject {subject_id}: {e}")

    if not all_epochs:
        print(f"No epochs found for subject {subject_id}.")
        return None

    print(f"Number of runs: {run_counter}")

    # Trun on the warnings
    warnings.filterwarnings("default", module="mne")

    return combined_epochs

## Concatecate

In [ ]:
subjects_epochs_list = []

for subject_id in subject_ids:
    subject_data = get_data_of_subject(subject_id=subject_id)
    if subject_data is not None:
        subjects_epochs_list.append(subject_data)
    else:
        print(f"Skipping subject {subject_id} due to issues loading data.")

Subjcet 01
Not setting metadata
4000 matching events found
No baseline correction applied
Adding metadata with 14 columns
Number of runs: 32
Subjcet 02
Not setting metadata
4000 matching events found
No baseline correction applied
Adding metadata with 14 columns
Number of runs: 32
Subjcet 04
Not setting metadata
4000 matching events found
No baseline correction applied
Adding metadata with 14 columns
Number of runs: 32
Subjcet 05
Not setting metadata
4000 matching events found
No baseline correction applied
Adding metadata with 14 columns
Number of runs: 32
Subjcet 06
Not setting metadata
4000 matching events found
No baseline correction applied
Adding metadata with 14 columns
Number of runs: 32


In [ ]:
if subjects_epochs_list:

    print(f"\nConcatenating epochs for {len(subjects_epochs_list)} subjects...")
    set_epochs = mne.concatenate_epochs(subjects_epochs_list)

    print("-"*40)
    print("Finished concatenating all epochs.")
    print(f"Estimated number of epochs: {estimeted_n_epochs == len(set_epochs)}")

else:
    print("No valid epochs found for any subject.")


Concatenating epochs for 5 subjects...
Adding metadata with 14 columns
20000 matching events found
No baseline correction applied
----------------------------------------
Finished concatenating all epochs.
Estimated number of epochs: True


## Save and Download

### Save

#### Data

In [ ]:
output_name = f'{input_ids_str}_set_epochs_data.npy'

# Save it to a .npy file
np.save(output_name, set_epochs.get_data())

In [ ]:
# output_name = f"{input_ids_str}_set_epochs.csv"

# # Convert the epochs object to a Pandas DataFrame
# epochs_df = set_epochs.to_data_frame()

# # Save the DataFrame to a CSV file
# epochs_df.to_csv(output_name, index=False)
# print(f"Epochs data saved to {output_name}")

#### Metadata

In [ ]:
assert isinstance(set_epochs.metadata, pd.DataFrame)
set_epochs.metadata.head()

meta_output_name = f'{input_ids_str}_set_epochs_metadata.csv'
set_epochs.metadata.to_csv(meta_output_name, index=False)

### Download

In [ ]:
files.download(output_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os

file_path = '/content/ds005811/firstset_epochs_data.npy'

if os.path.exists(file_path):
    size_in_bytes = os.path.getsize(file_path)
    size_in_mb = size_in_bytes / (1024 * 1024)
    size_in_gb = size_in_bytes / (1024 * 1024 * 1024)
    print(f"The file '{os.path.basename(file_path)}' is {size_in_bytes} bytes ({size_in_mb:.2f} MB or {size_in_gb:.2f} GB) large.")
else:
    print(f"File not found: {file_path}")

The file 'firstset_epochs_data.npy' is 2314240128 bytes (2207.03 MB or 2.16 GB) large.


In [ ]:
output_name_compressed = f'{input_ids_str}_set_epochs_data_compressed.npz'

# Save with compression
np.savez_compressed(output_name_compressed, data=set_epochs.get_data())

print(f'Saved compressed data to {output_name_compressed}')

compressed_path = '/content/firstset_epochs_data_compressed.npz'

# Check the size of the compressed file
size_compressed_bytes = os.path.getsize(output_name_compressed)
size_compressed_gb = size_compressed_bytes / (1024 * 1024 * 1024)
print(f"The file '{os.path.basename(output_name_compressed)}' is {size_compressed_bytes} bytes ({size_compressed_gb:.2f} GB) large.")

# You can load it back using:
# loaded_data = np.load(output_name_compressed)['data']


Saved compressed data to first_set_epochs_data_compressed.npz
The file 'first_set_epochs_data_compressed.npz' is 2110347166 bytes (1.97 GB) large.
